# M02 · Quadrature encoders and velocity from a wrapping counter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/04_control_motion/M02_quadrature_encoders_and_velocity_from_a_wrapping.ipynb)

**Track:** Control and motion · **Skill axis (sheet):** Control and motion · **Time:** about 50 min

**Skill Hub concepts:** Quadrature decoding; Signed modular velocity

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("M02")

## Map: two square waves 90 degrees apart

```
 A  __|‾‾‾‾|____|‾‾‾‾|____          state = (A << 1) | B
 B  ____|‾‾‾‾|____|‾‾‾‾|__          forward: 00 -> 01 -> 11 -> 10 -> 00   (each arrow = +1 count, "x4")
```

| prev \ new | 00 | 01 | 11 | 10 |
|---|---|---|---|---|
| **00** | 0 | +1 | err | -1 |
| **01** | -1 | 0 | +1 | err |
| **11** | err | -1 | 0 | +1 |
| **10** | +1 | err | -1 | 0 |

"err" = both lines changed between samples: you sampled too slowly or saw noise. A hardware timer in encoder mode does
this table for you and counts in a 16- or 32-bit register that **wraps**.

Signed modular difference on an $n$-bit counter:
$$\Delta = \big((c_{now} - c_{prev} + 2^{n-1}) \bmod 2^{n}\big) - 2^{n-1}$$
(in C: `(int16_t)(now - prev)`), valid while the true motion per sample is under half the counter range.

Speed: $\text{rpm} = \dfrac{\Delta}{\text{CPR}} \cdot \dfrac{60}{\Delta t}$ with CPR = 4 x lines for x4 decoding.

## Task 1: a software x4 decoder

In [ ]:
class QuadDecoder:
    TABLE = {(0, 1): +1, (1, 3): +1, (3, 2): +1, (2, 0): +1,
             (1, 0): -1, (3, 1): -1, (2, 3): -1, (0, 2): -1}

    def __init__(self):
        self.prev, self.position, self.errors = 0, 0, 0

    def update(self, a, b):
        raise NotImplementedError  # TODO

lab.check("decoder", QuadDecoder)

In [ ]:
%%writefile m02_decoder.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/m02.h
// Grade it with lab.check("decoder", "m02_decoder.c"). For C++, name the file .cpp in both places.
#include "m02.h"

void qd_init(QuadDecoder *d) { d->prev = 0; d->position = 0; d->errors = 0; }

int32_t qd_update(QuadDecoder *d, int a, int b) {
    (void)a; (void)b;
    return d->position; /* TODO: a 4x4 transition table indexed by previous and new state */
}

In [ ]:
# lab.check("decoder", "m02_decoder.c")  # uncomment after writing the C/C++ cell above

## Task 2: velocity from a wrapping counter

In [ ]:
def signed_delta(now, prev, bits=16):
    raise NotImplementedError  # TODO

def rpm(delta_counts, dt, lines):
    raise NotImplementedError  # TODO

lab.check("velocity", signed_delta, rpm)

In [ ]:
%%writefile m02_velocity.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/m02.h
// Grade it with lab.check("velocity", "m02_velocity.c"). For C++, name the file .cpp in both places.
#include "m02.h"

int64_t signed_delta(uint64_t now, uint64_t prev, int bits) {
    (void)now; (void)prev; (void)bits;
    return 0; /* TODO: modular difference, then map the upper half to negative */
}

double rpm(int64_t delta_counts, double dt, uint32_t lines) { (void)delta_counts; (void)dt; (void)lines; return 0; /* TODO */ }

In [ ]:
# lab.check("velocity", "m02_velocity.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** A 1024-line encoder decoded x4. Counts per revolution?

**M2 (Apply).** 16-bit counter: previous 65530, now 6. Signed delta?

**M3 (Apply).** 1000 counts in 10 ms on the 1024-line encoder. Speed in rpm?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why decode x4 instead of counting only rising edges of A?
- a) it is required by the timer
- b) four times the resolution from the same encoder, and direction comes for free from the state order
- c) it removes the need for debouncing

**K2 (Analyze).** At 2 rpm a 1 kHz loop sees 0 or 1 count per sample, so the speed estimate jumps between 0 and
14.6 rpm. Explain why and two better estimators.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `m02.h`).
3. In `roboprep/checks/m02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()